In [58]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

In [59]:
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [60]:
df = pd.read_csv(r"C:/Users/SAKTHI/Desktop/rework/data/fashion-mnist_train.csv")
print("Dataset shape:",df.shape)
print("total Image size:",len(df))

Dataset shape: (60000, 785)
total Image size: 60000


In [61]:
CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

NUM_CLASSES = len(CLASS_NAMES)
print("Number of classes:", NUM_CLASSES)

Number of classes: 10


In [62]:
x = df.iloc[:,1:].values

y = df.iloc[:,0].values

print(f"image pixel : {len(x[0])} | label : {np.size(y[0])}")

image pixel : 784 | label : 1


In [63]:
# split the train dataset into train - 80 , val - 20

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

print(f"train image:{len(x_train)} | train label {len(y_train)}")
print(f"Val image:{len(x_val)} | val label:{len(y_val)}")


train image:48000 | train label 48000
Val image:12000 | val label:12000


In [64]:
weights = models.ResNet50_Weights.IMAGENET1K_V1

print(weights.transforms())

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [65]:
image_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
				)
])

In [66]:
# custom dataset for turn pixel into an image , 1 channel to 3 channel for pretrained model

class CustomDataset(Dataset):

    def __init__(self, images, labels, transform):
        # Store raw images, labels and preprocessing
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        # Number of images available in the dataset
        return len(self.images)

    def __getitem__(self, idx):
        
        # Select one image using its index
        image = self.images[idx]
        image = image.reshape(28, 28).astype(np.uint8)
        image = np.stack([image] * 3, axis=-1)
        # Convert NumPy array into PIL image
        image = Image.fromarray(image)
        image = self.transform(image)
        label = torch.tensor(self.labels[idx],dtype=torch.long)
        return image, label

In [67]:
train_dataset = CustomDataset(x_train, y_train, image_transform)
val_dataset = CustomDataset(x_val, y_val, image_transform)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False, pin_memory=True)
val_loader  = DataLoader(val_dataset, batch_size=32, shuffle=False, pin_memory=True)

Train dataset: 48000
Validation dataset: 12000


In [68]:
feature_extractor = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

for param in feature_extractor.parameters():
    param.requires_grad = False

# removing the orignial clasifier to get the extracted feature

feature_extractor.fc = nn.Identity()

feature_extractor = feature_extractor.to(device)

feature_extractor.eval()

print(feature_extractor.fc)


Identity()


In [69]:
# Check the feature shape using one batch before extracting the full dataset.
images, labels = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    features = feature_extractor(images)

print("Input image batch shape :", images.shape)
print("Extracted feature shape :", features.shape)

Input image batch shape : torch.Size([32, 3, 224, 224])
Extracted feature shape : torch.Size([32, 2048])


In [70]:
# extracts features from the pretrained model but does not train the model

def extract_features(model, loader, device):
    model.eval()

    all_features = []
    all_labels = []

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(loader):

            images = images.to(device)

            #image into features.
            features = model(images)

            # Move results to CPU before storing them.
            
            all_features.append(features.cpu())
            all_labels.append(labels.cpu())

            if (batch_idx + 1) % 100 == 0:
                print(f"Processed {batch_idx + 1}/{len(loader)} batches")

    # Combine all batches into one tensor
    
    all_features = torch.cat(all_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_features, all_labels


In [71]:
from pathlib import Path

# Create folder
FEATURE_DIR = Path("saved_features")
FEATURE_DIR.mkdir(exist_ok=True)

# Extract features
train_features, train_labels = extract_features(feature_extractor,train_loader,device)
val_features, val_labels = extract_features(feature_extractor,val_loader,device)

# Save features
torch.save({"features": train_features, "labels": train_labels},FEATURE_DIR / "resnet_train_features.pt")
torch.save({"features": val_features, "labels": val_labels},FEATURE_DIR / "resnet_val_features.pt")


Processed 100/1500 batches
Processed 200/1500 batches
Processed 300/1500 batches
Processed 400/1500 batches
Processed 500/1500 batches
Processed 600/1500 batches
Processed 700/1500 batches
Processed 800/1500 batches
Processed 900/1500 batches
Processed 1000/1500 batches
Processed 1100/1500 batches
Processed 1200/1500 batches
Processed 1300/1500 batches
Processed 1400/1500 batches
Processed 1500/1500 batches
Processed 100/375 batches
Processed 200/375 batches
Processed 300/375 batches


In [72]:
from torch.utils.data import TensorDataset, DataLoader

train_feature_dataset = TensorDataset(train_features,train_labels)
val_feature_dataset = TensorDataset(val_features,val_labels)

train_feature_loader = DataLoader(train_feature_dataset,batch_size=128,shuffle=True)
val_feature_loader = DataLoader(val_feature_dataset,batch_size=128,shuffle=False)

In [73]:
classifier = nn.Sequential(
    nn.Linear(2048, 512),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(256, 10)
)

classifier = classifier.to(device)

In [74]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=1e-4)

In [75]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "resnet50_best.pth"

best_val_accuracy = 0.0


In [77]:
EPOCHS = 15

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []


for epoch in range(EPOCHS):

    # TRAINING
    classifier.train()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for features, labels in train_feature_loader:

        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = classifier(features)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * features.size(0)

        predictions = outputs.argmax(dim=1)
        running_correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_accuracy = running_correct / total

    # VALIDATION
    classifier.eval()

    val_running_loss = 0.0
    val_running_correct = 0
    val_total = 0

    with torch.no_grad():
        for features, labels in val_feature_loader:

            features = features.to(device)
            labels = labels.to(device)

            outputs = classifier(features)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * features.size(0)

            predictions = outputs.argmax(dim=1)
            val_running_correct += (predictions == labels).sum().item()

            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_accuracy = val_running_correct / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(
            classifier.state_dict(),
            MODEL_PATH
        )

        print(
            f"Best model saved | "
            f"Val Accuracy: {best_val_accuracy:.4f}"
        )


Epoch [1/15] | Train Loss: 0.2583 | Train Acc: 0.9056 | Val Loss: 0.2657 | Val Acc: 0.9034
Best model saved | Val Accuracy: 0.9034
Epoch [2/15] | Train Loss: 0.2514 | Train Acc: 0.9076 | Val Loss: 0.2748 | Val Acc: 0.8986
Epoch [3/15] | Train Loss: 0.2446 | Train Acc: 0.9098 | Val Loss: 0.2570 | Val Acc: 0.9066
Best model saved | Val Accuracy: 0.9066
Epoch [4/15] | Train Loss: 0.2394 | Train Acc: 0.9129 | Val Loss: 0.2548 | Val Acc: 0.9083
Best model saved | Val Accuracy: 0.9083
Epoch [5/15] | Train Loss: 0.2338 | Train Acc: 0.9143 | Val Loss: 0.2680 | Val Acc: 0.9017
Epoch [6/15] | Train Loss: 0.2252 | Train Acc: 0.9180 | Val Loss: 0.2474 | Val Acc: 0.9093
Best model saved | Val Accuracy: 0.9093
Epoch [7/15] | Train Loss: 0.2222 | Train Acc: 0.9185 | Val Loss: 0.2507 | Val Acc: 0.9083
Epoch [8/15] | Train Loss: 0.2162 | Train Acc: 0.9207 | Val Loss: 0.2483 | Val Acc: 0.9097
Best model saved | Val Accuracy: 0.9097
Epoch [9/15] | Train Loss: 0.2116 | Train Acc: 0.9234 | Val Loss: 0.2479